# MedLens: Fine-tuning Gemma 4 E2B for Drug Interaction Detection

<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
</div>

This notebook fine-tunes **Gemma 4 E2B** (text-only) for polypharmacy drug interaction detection using the MedLens training dataset.


## Contents:
1. [Installation](#Installation)
2. [Load Model](#Load-Model)
3. [Prepare Data](#Prepare-Data)
4. [Train](#Train)
5. [Inference](#Inference)
6. [Save Model](#Save-Model)

## Installation

In [1]:
%%capture
# import os, re
# if "COLAB_" not in "".join(os.environ.keys()):
#     # Local installation
#     !pip install unsloth
# else:
#     # Colab installation
#     import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
#     xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
#     !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
#     !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
# !pip install --no-deps transformers==5.5.0
import torch; torch._dynamo.config.cache_size_limit = 64;  # Or your desired limit


## Load Model

Load Gemma 4 E2B with 4-bit quantization and add LoRA adapters.

In [2]:
from unsloth import FastModel
import torch

# Configuration
MODEL_NAME = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True  # Use False for 16-bit LoRA (more accurate but more VRAM)

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto detect
    load_in_4bit=LOAD_IN_4BIT,
    full_finetuning=False,
    # token="YOUR_HF_TOKEN",  # Uncomment if needed
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/ashu/github/MedLens/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4070 Ti SUPER. Num GPUs = 1. Max memory: 15.992 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 2011/2011 [01:29<00:00, 22.56it/s] 


In [3]:
# Add LoRA adapters
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,   # Text only - no vision
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=32,                # LoRA rank - larger = higher accuracy but might overfit
    lora_alpha=32,       # Recommended alpha == r at least
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

In [4]:
# Apply Gemma-4 chat template
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-4",  # Use "gemma-4-thinking" if you want thinking mode
)

## Prepare Data

Load the MedLens training dataset and format it for training.

**If running on Colab:** Upload `medlens_train.jsonl` to the Colab runtime or mount Google Drive.

In [5]:
import json

# For Colab: upload the file or mount drive
DATASET_PATH = "medlens_train.jsonl"  # Adjust path as needed

# Optional: Upload from your machine (Colab only)
# from google.colab import files
# uploaded = files.upload()

# Optional: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_PATH = "/content/drive/MyDrive/medlens_train.jsonl"

def load_medlens_dataset(path):
    """Load JSONL dataset."""
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line.strip())
            data.append(item)
    print(f"Loaded {len(data)} training examples")
    return data

raw_data = load_medlens_dataset(DATASET_PATH)

Loaded 5000 training examples


In [6]:
# Look at a sample
print("Sample training example:")
print(json.dumps(raw_data[0], indent=2))

Sample training example:
{
  "messages": [
    {
      "role": "user",
      "content": "I'm a 55 year-old male. I currently take: carbimazole, mycophenolate mofetil, tacrolimus. I take them for Product used for unknown indication, Prophylaxis against transplant rejection. I've been experiencing: Agranulocytosis. Are any of these drugs interacting?"
    },
    {
      "role": "assistant",
      "content": "<|think|>Patient (age: 55 year-old, sex: male) on 3 concurrent drugs: carbimazole, mycophenolate mofetil, tacrolimus. 3 possible pairwise interactions to evaluate. Reported adverse events: Agranulocytosis. Demographics are relevant \u2014 elderly and pediatric patients have altered pharmacokinetics, and sex-based differences affect CYP450 metabolism. This pattern matches FAERS-documented multi-drug cases with severity classification: Major. Systematic pair check needed before giving advice.</think>\n\n\ud83d\udea8 **MAJOR interaction risk detected**\n\n**Medications reviewed (3):** c

In [12]:
from datasets import Dataset

def format_for_training(examples, tokenizer):
    """Convert messages format to a Hugging Face Dataset using the Gemma-4 chat template."""
    formatted = []
    for example in examples:
        messages = example["messages"]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        ).removeprefix("<bos>")  # Remove <bos> as processor will add it
        formatted.append({"text": text, "messages": messages})
    return Dataset.from_list(formatted)

dataset = format_for_training(raw_data, tokenizer)
print(f"Formatted {len(dataset)} examples for training")
print(f"Dataset type: {type(dataset).__name__}")

Formatted 5000 examples for training
Dataset type: Dataset


In [13]:
# Preview formatted text
print("Formatted text sample:")
print("-" * 50)
print(dataset[0]["text"][:1000])
print("-" * 50)

Formatted text sample:
--------------------------------------------------
<|turn>user
I'm a 55 year-old male. I currently take: carbimazole, mycophenolate mofetil, tacrolimus. I take them for Product used for unknown indication, Prophylaxis against transplant rejection. I've been experiencing: Agranulocytosis. Are any of these drugs interacting?<turn|>
<|turn>model
<|think|>Patient (age: 55 year-old, sex: male) on 3 concurrent drugs: carbimazole, mycophenolate mofetil, tacrolimus. 3 possible pairwise interactions to evaluate. Reported adverse events: Agranulocytosis. Demographics are relevant — elderly and pediatric patients have altered pharmacokinetics, and sex-based differences affect CYP450 metabolism. This pattern matches FAERS-documented multi-drug cases with severity classification: Major. Systematic pair check needed before giving advice.</think>

🚨 **MAJOR interaction risk detected**

**Medications reviewed (3):** carbimazole, mycophenolate mofetil, tacrolimus

**Reported adve

In [17]:
dataset[0]

{'text': "<|turn>user\nI'm a 55 year-old male. I currently take: carbimazole, mycophenolate mofetil, tacrolimus. I take them for Product used for unknown indication, Prophylaxis against transplant rejection. I've been experiencing: Agranulocytosis. Are any of these drugs interacting?<turn|>\n<|turn>model\n<|think|>Patient (age: 55 year-old, sex: male) on 3 concurrent drugs: carbimazole, mycophenolate mofetil, tacrolimus. 3 possible pairwise interactions to evaluate. Reported adverse events: Agranulocytosis. Demographics are relevant — elderly and pediatric patients have altered pharmacokinetics, and sex-based differences affect CYP450 metabolism. This pattern matches FAERS-documented multi-drug cases with severity classification: Major. Systematic pair check needed before giving advice.</think>\n\n🚨 **MAJOR interaction risk detected**\n\n**Medications reviewed (3):** carbimazole, mycophenolate mofetil, tacrolimus\n\n**Reported adverse events:** Agranulocytosis\n\nBased on real-world ph

## Train

Set up the trainer and run fine-tuning. Uses response-only training to focus on assistant outputs.

In [16]:
from trl import SFTTrainer, SFTConfig

# Training configuration
MAX_STEPS = 60              # Set to -1 for full training run, 60 for quick test
NUM_TRAIN_EPOCHS = None     # Set to 1 (or more) for full training run
LEARNING_RATE = 2e-4        # Reduce to 2e-5 for longer runs

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,  # Effective batch size = 4
        warmup_ratio=0.03,
        max_steps=MAX_STEPS,            # Use -1 for epoch-based training
        # num_train_epochs=NUM_TRAIN_EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=10,
        save_strategy="steps",
        save_steps=200,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        max_grad_norm=0.3,
        seed=3407,
        output_dir="outputs",
        report_to="none",  # Change to "wandb" for W&B logging
        max_length=MAX_SEQ_LENGTH,
    ),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Unsloth: Tokenizing ["text"] (num_proc=22): 100%|██████████| 5000/5000 [00:46<00:00, 106.56 examples/s]


In [18]:
# Train only on assistant responses (not user inputs)
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)

Filter (num_proc=22): 100%|██████████| 5000/5000 [00:00<00:00, 7727.18 examples/s] 


In [19]:
# Verify masking is working - should show only assistant response
print("Masked labels (should show only assistant response):")
print("-" * 50)
labels_sample = trainer.train_dataset[0]["labels"]
print(tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in labels_sample]).replace(tokenizer.pad_token, " ")[:500])
print("-" * 50)

Masked labels (should show only assistant response):
--------------------------------------------------
                                                                          <|think|>Patient (age: 55 year-old, sex: male) on 3 concurrent drugs: carbimazole, mycophenolate mofetil, tacrolimus. 3 possible pairwise interactions to evaluate. Reported adverse events: Agranulocytosis. Demographics are relevant — elderly and pediatric patients have altered pharmacokinetics, and sex-based differences affect CYP450 metabolism. This pattern matches FAERS-documented multi-drug cases with severity classific
--------------------------------------------------


In [20]:
# Show GPU memory before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 4070 Ti SUPER. Max memory = 15.992 GB.
7.141 GB of memory reserved.


In [21]:
# Train!
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 50,675,712 of 5,173,853,728 (0.98% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Step,Training Loss
10,8.512742
20,3.059754
30,1.489769
40,1.264665
50,1.153474
60,1.104135


In [28]:
# Show training stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)

print(f"\n{'='*50}")
print(f"Training completed!")
print(f"{trainer_stats.metrics['train_runtime']:.2f} seconds used for training.")
print(f"{trainer_stats.metrics['train_runtime']/60:.2f} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"{'='*50}")


Training completed!
179.15 seconds used for training.
2.99 minutes used for training.
Peak reserved memory = 10.707 GB.
Peak reserved memory for training = 3.566 GB.
Peak reserved memory % of max memory = 66.952 %.


## Inference

Test the fine-tuned model with sample drug interaction queries.

In [26]:
from transformers import TextStreamer

def query_model(prompt, max_new_tokens=256):
    """Query the model with a drug interaction prompt."""
    messages = [
        {
            "role": "user",
            "content": [{"type": "text", "text": prompt}],
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        tokenize=True,
        return_dict=True,
    ).to("cuda")

    text_streamer = TextStreamer(tokenizer, skip_prompt=True)

    _ = model.generate(
        **inputs,
        streamer=text_streamer,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        # Recommended Gemma-4 settings
        temperature=1.0,
        top_p=0.95,
        top_k=64,
    )

In [27]:
# Test 1: Classic anticoagulant interaction
test_prompt_1 = """I'm a 65 year-old male. I currently take: aspirin, clopidogrel, warfarin. 
I take them for atrial fibrillation and coronary artery disease. 
I've been experiencing: bruising easily, nosebleeds. 
Are any of these drugs interacting?"""

print("Query 1:")
print(test_prompt_1)
print("\nResponse:")
query_model(test_prompt_1)

Query 1:
I'm a 65 year-old male. I currently take: aspirin, clopidogrel, warfarin. 
I take them for atrial fibrillation and coronary artery disease. 
I've been experiencing: bruising easily, nosebleeds. 
Are any of these drugs interacting?

Response:
I am an AI, and **I cannot provide medical advice or diagnose drug interactions.** This is a very important matter that requires evaluation by a qualified healthcare professional, such as your doctor or pharmacist.

**Please contact your prescribing physician or pharmacist immediately** to discuss your symptoms (easy bruising and nosebleeds) in the context of your current medications (aspirin, clopidogrel, warfarin). They are the only ones who can safely assess your medical situation and advise you on any potential interactions or necessary changes to your treatment plan.

**Why this is urgent:**

* **Warfarin** is a medication with a narrow therapeutic window, meaning small changes in dose or other factors can significantly increase the r

In [ ]:
# Test 2: NSAID + ACE inhibitor + diuretic (triple whammy)
test_prompt_2 = """I'm a 72 year-old female. I currently take: ibuprofen, lisinopril, furosemide.
I take them for arthritis pain, hypertension, and heart failure.
I've been experiencing: dizziness, decreased urination, swelling in legs.
Are any of these drugs interacting?"""

print("Query 2:")
print(test_prompt_2)
print("\nResponse:")
query_model(test_prompt_2)

In [ ]:
# Test 3: Multiple CNS depressants
test_prompt_3 = """I'm a 45 year-old male. I currently take: oxycodone, alprazolam, gabapentin, zolpidem.
I take them for chronic back pain, anxiety, and insomnia.
I've been experiencing: excessive drowsiness, confusion, memory problems.
Are any of these drugs interacting?"""

print("Query 3:")
print(test_prompt_3)
print("\nResponse:")
query_model(test_prompt_3)

## Save Model

Multiple options for saving the fine-tuned model:
1. **LoRA adapters only** (~50-100MB) - smallest, requires base model to load
2. **Merged model (float16)** - for VLLM, HuggingFace deployment
3. **GGUF format** - for llama.cpp, Ollama, local inference

In [ ]:
# Option 1: Save LoRA adapters only (smallest size)
LORA_OUTPUT_DIR = "medlens_lora"

model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)
print(f"LoRA adapters saved to {LORA_OUTPUT_DIR}")

In [ ]:
# Option 2: Save merged model in float16 (for VLLM, deployment)
SAVE_MERGED = False  # Set to True to save

if SAVE_MERGED:
    MERGED_OUTPUT_DIR = "medlens_merged"
    model.save_pretrained_merged(MERGED_OUTPUT_DIR, tokenizer)
    print(f"Merged model saved to {MERGED_OUTPUT_DIR}")

In [ ]:
# Option 3: Save as GGUF for llama.cpp / Ollama
SAVE_GGUF = False  # Set to True to save
GGUF_QUANT = "Q8_0"  # Options: "Q8_0", "BF16", "F16"

if SAVE_GGUF:
    GGUF_OUTPUT_DIR = "medlens_gguf"
    model.save_pretrained_gguf(
        GGUF_OUTPUT_DIR,
        tokenizer,
        quantization_method=GGUF_QUANT,
    )
    print(f"GGUF ({GGUF_QUANT}) saved to {GGUF_OUTPUT_DIR}")

In [ ]:
# Option 4: Push to Hugging Face Hub
PUSH_TO_HUB = False  # Set to True to push
HF_REPO = "YOUR_USERNAME/medlens-gemma4-e2b"  # Change this!
HF_TOKEN = "YOUR_HF_TOKEN"  # Get from https://huggingface.co/settings/tokens

if PUSH_TO_HUB:
    # Push LoRA adapters
    model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f"Pushed to {HF_REPO}")
    
    # Or push merged model:
    # model.push_to_hub_merged(HF_REPO + "-merged", tokenizer, token=HF_TOKEN)

## Load Saved LoRA Adapters

Example of loading the saved LoRA adapters for inference.

In [ ]:
# To load saved LoRA adapters later:
LOAD_SAVED = False  # Set to True to load

if LOAD_SAVED:
    from unsloth import FastModel
    
    model, tokenizer = FastModel.from_pretrained(
        model_name="medlens_lora",  # Path to saved LoRA
        max_seq_length=4096,
        load_in_4bit=True,
    )
    
    from unsloth.chat_templates import get_chat_template
    tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
    
    print("Loaded fine-tuned model from medlens_lora")

---

## Done!

Your fine-tuned MedLens model is ready. Next steps:

1. **Evaluate** on held-out DDI Corpus benchmark
2. **Export to LiteRT** for Android deployment
3. **Integrate** with MedLens Android app

For questions, see [Unsloth Discord](https://discord.gg/unsloth) or [MedLens repo](https://github.com/your-repo/medlens).